In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np

In [16]:
# Create a DataFrame from your provided data
data = {
    "Batch Size": [16, 32, 64, 128],
    "ACC": [0.9130, 0.9216, 0.9790, 0.9839],
    "AUC": [0.9875, 0.9859, 0.9857, 0.9850],
    "PRE": [0.9093, 0.8966, 0.9066, 0.9006],
    "SP": [0.9062, 0.8938, 0.9029, 0.8963],
    "SN": [0.9713, 0.9674, 0.9040, 0.9680],
    "F1": [0.9073, 0.8948, 0.8795, 0.8979],
    "MCC": [0.8835, 0.8681, 0.9790, 0.8712]
}

In [17]:
df = pd.DataFrame(data)

In [18]:
# Reset matplotlib settings to default
plt.rcParams.update(plt.rcParamsDefault)

# Set Times New Roman font with fallback
plt.rcParams['font.family'] = ['Times New Roman', 'serif']
plt.rcParams['mathtext.fontset'] = 'stix'

# Set global plot style with large font sizes
plt.rcParams['font.size'] = 32
plt.rcParams['axes.labelsize'] = 36
plt.rcParams['axes.titlesize'] = 40
plt.rcParams['xtick.labelsize'] = 30
plt.rcParams['ytick.labelsize'] = 30
plt.rcParams['legend.fontsize'] = 22
plt.rcParams['figure.dpi'] = 1000
plt.rcParams['savefig.dpi'] = 1000
plt.rcParams['figure.facecolor'] = 'white'

In [19]:
# Enhanced color palette for Batch Size
batch_size_palette = {
    16: '#1F77B4',   # Bright Blue
    32: '#FF7F0E',   # Rich Orange
    64: '#2CA02C',   # Vivid Green
    128: '#D62728'   # Bold Red
}

# Metric palette
metric_palettes = {
    'ACC': 'viridis',
    'AUC': 'plasma',
    'PRE': 'cividis',
    'SP': 'magma',
    'SN': 'inferno',
    'F1': 'cividis',
    'MCC': 'plasma'
}

In [20]:
# Function to save figures in both PNG and PDF
def save_figure(fig, filename):
    output_folder = "Experiment_Output_Figures/Batch_Size"
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    png_path = os.path.join(output_folder, f"{filename}.png")
    pdf_path = os.path.join(output_folder, f"{filename}.pdf")
    
    fig.savefig(png_path, dpi=1000, bbox_inches='tight', facecolor='white', format='png')
    fig.savefig(pdf_path, dpi=1000, bbox_inches='tight', facecolor='white', format='pdf')
    
    print(f"Saved: {png_path} and {pdf_path}")
    plt.close(fig)

In [31]:
# Function to generate the violin plot visualization
def generate_violin_plot():
    metrics = ['ACC', 'AUC', 'PRE', 'SP', 'SN', 'F1', 'MCC']
    melted_df = pd.melt(df, id_vars=['Batch Size'],
                       value_vars=metrics,
                       var_name='Metric',
                       value_name='Score')
    
    fig, ax = plt.subplots(figsize=(20, 14))
    sns.violinplot(x='Batch Size', 
                   y='Score', 
                   hue='Batch Size',
                   data=melted_df,
                   palette=batch_size_palette, 
                   inner='box',
                   linewidth=2, 
                   ax=ax,
                   legend=False)
    
    ax.set_title('Distribution of Performance Scores by Batch Size',
                fontsize=44, pad=30)
    ax.set_xlabel('Batch Size', fontsize=40, labelpad=25)
    ax.set_ylabel('Score Distribution Across Metrics', fontsize=40, labelpad=25)
    ax.set_ylim(0.8, 1.1)  # Adjusted for your data range
    
    plt.tight_layout(pad=3.0)
    save_figure(fig, "batch_size_violin_plot")

In [22]:
# Function to generate the timing comparison plot
def generate_timing_comparison():
    fig, ax = plt.subplots(figsize=(18, 10))
    sorted_df = df.sort_values(by='Training_Time', ascending=True)
    
    bars = ax.barh(sorted_df['Batch Size'].astype(str), sorted_df['Training_Time'],
                  color=[batch_size_palette[bs] for bs in sorted_df['Batch Size']],
                  height=0.6)
    
    for i, batch_size in enumerate(sorted_df['Batch Size']):
        test_time = sorted_df[sorted_df['Batch Size'] == batch_size]['Testing_Time'].values[0]
        ax.text(10, i, f'Test: {test_time:.4f}s', ha='left', va='center',
                fontsize=22, color='black')
    
    ax.set_title('Training and Testing Time by Batch Size', fontsize=40, pad=30)
    ax.set_xlabel('Training Time (seconds)', fontsize=36, labelpad=25)
    ax.set_ylabel('Batch Size', fontsize=36, labelpad=25)
    
    max_time = sorted_df['Training_Time'].max()
    ax.set_xlim(0, max_time * 1.2)
    
    plt.tight_layout(pad=3.0)
    save_figure(fig, "batch_size_timing_comparison")

In [23]:
metrics = ['ACC', 'AUC']
melted_df = pd.melt(df, id_vars=['Batch Size'],
                   value_vars=metrics,
                   var_name='Metric',
                   value_name='Score')

# Grouped bar chart
fig, ax = plt.subplots(figsize=(20, 14))
sns.barplot(x='Metric', y='Score', hue='Batch Size',
            data=melted_df, palette=batch_size_palette,
            ax=ax, edgecolor='none')

ax.set_title('Comparison of Batch Sizes across Metrics', fontsize=44, pad=30)
ax.set_xlabel('Evaluation Metric', fontsize=40, labelpad=25)
ax.set_ylabel('Score', fontsize=40, labelpad=25)
ax.set_ylim(0.9, 1.01)  # Adjusted for your data range

ax.legend(title='Batch Size', title_fontsize=24, fontsize=22,
          bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout(pad=3.0)
save_figure(fig, "batch_size_comparison_grouped_bar")

Saved: Experiment_Output_Figures/Batch_Size/batch_size_comparison_grouped_bar.png and Experiment_Output_Figures/Batch_Size/batch_size_comparison_grouped_bar.pdf


In [24]:
# Radar Chart
categories = metrics
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(16, 16), subplot_kw=dict(polar=True))
for i, batch_size in enumerate(df['Batch Size']):
    values = df.loc[i, metrics].values.tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=4, label=str(batch_size),
            color=batch_size_palette[batch_size])
    ax.fill(angles, values, alpha=0.1, color=batch_size_palette[batch_size])

ax.set_ylim(0.9, 1.01)  # Adjusted for your data range
plt.xticks(angles[:-1], categories, fontsize=36)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0), fontsize=22)
plt.title('Batch Size Comparison (Radar Chart)',
          fontsize=44, pad=40, y=1.08)

plt.tight_layout(pad=3.0)
save_figure(fig, "batch_size_radar_chart")

Saved: Experiment_Output_Figures/Batch_Size/batch_size_radar_chart.png and Experiment_Output_Figures/Batch_Size/batch_size_radar_chart.pdf


In [25]:
# Heatmap
heatmap_df = df[['Batch Size'] + metrics].set_index('Batch Size')
fig, ax = plt.subplots(figsize=(18, 12))

heatmap = sns.heatmap(heatmap_df, annot=True, fmt=".4f", cmap="YlGnBu",
                     linewidths=0.5, linecolor='white', annot_kws={'size': 26},
                     vmin=0.9, vmax=1.0)  # Adjusted for your data range

cbar = heatmap.collections[0].colorbar
cbar.set_label('Score', size=34)
cbar.ax.tick_params(labelsize=28)

ax.set_title('Batch Size Performance Heatmap', fontsize=40, pad=30)
ax.set_xlabel('Evaluation Metrics', fontsize=36, labelpad=25)
ax.set_ylabel('Batch Size', fontsize=36, labelpad=25)

plt.tight_layout(pad=3.0)
save_figure(fig, "batch_size_heatmap")

Saved: Experiment_Output_Figures/Batch_Size/batch_size_heatmap.png and Experiment_Output_Figures/Batch_Size/batch_size_heatmap.pdf


In [32]:
generate_violin_plot()

Saved: Experiment_Output_Figures/Batch_Size/batch_size_violin_plot.png and Experiment_Output_Figures/Batch_Size/batch_size_violin_plot.pdf


In [27]:
# generate_timing_comparison()

In [28]:
print("All batch size visualizations have been generated!")
print(f"Files are saved in the 'Batch_Size' folder")

All batch size visualizations have been generated!
Files are saved in the 'Batch_Size' folder
